# Notebook 0: Data Preparation for Target Variables "Milieuschutz"

This notebook builds the target variable for Module 2, the classification: whether a Berlin planning area (PLR) is under Milieuschutz (Erhaltungsverordnung § 172 Abs. 1 Nr. 2 BauGB). 

Protected-area polygons are loaded from the Berlin WFS and intersected with the 542 PLR via an area-weighted spatial overlay, yielding two labels per PLR: milieu_anteil, the continuous share of PLR area under protection (0–1), and milieu_majoritaet, a binary flag set to 1 where more than 50% of the area is protected. 

PLR with no overlap are assigned an observed zero rather than treated as missing, since absence of designation is real information under the positive-unlabeled framing. The binary label is the model target; the continuous share is retained for the regression cross-check.

In [1]:
#import the libraries necessary to load data 

import geopandas as gpd
import pandas as pd
import numpy as np


In [ ]:
#Produces two labels per planning area:
#milieu_anteil: the continuous share of PLR area under Milieuschutz 
#milieu_majoritaet: binary, 1 if >50% of PLR area protected 

WORKING_CRS = 25833

#Load Milieuschutzgebiete from the Berlin WFS
#erhaltgeb_em = "Erhaltung der Zusammensetzung der Wohnbevölkerung" = Milieuschutz

url = ("https://gdi.berlin.de/services/wfs/erhaltungsverordnungsgebiete"
       "?service=WFS&version=2.0.0&request=GetFeature"
       "&typeNames=erhaltungsverordnungsgebiete:erhaltgeb_em"
       "&srsName=EPSG:25833")
milieu = gpd.read_file(url).to_crs(WORKING_CRS)

#repair invalid geometries
milieu["geometry"] = milieu.geometry.buffer(0)          
print("Milieuschutzgebiete geladen:", milieu.shape)
print("Spalten:", milieu.columns.tolist())

#Load PLR backbone (same file as the other variables)
plr = gpd.read_file("../../data/real_estate/GeoData/PLR_raw.geojson").to_crs(WORKING_CRS)
plr["geometry"] = plr.geometry.buffer(0)
plr["plr_id"] = plr["plr_id"].astype(str).str.zfill(8)

#calculate area per PLR (m²)
plr_area = plr.set_index("plr_id").geometry.area        
print("PLR geladen:", plr.shape[0])

#Intersect protected areas with PLR, sum protected area per PLR
inter = gpd.overlay(
    milieu[["geometry"]],
    plr[["plr_id", "geometry"]],
    how="intersection", keep_geom_type=True,
)
inter["area"] = inter.geometry.area
inter = inter[inter["area"] > 0]

#protected m² per PLR
geschuetzt = inter.groupby("plr_id")["area"].sum()       

#Build the master table on all 542 PLR
ziel = plr[["plr_id", "plr_name", "bez"]].copy()

#variable 1 — continuous share of PLR area under Milieuschutz (0..1)
# PLR with no overlap -> 0 (observed, not missing)

ziel["milieu_anteil"] = (
    ziel["plr_id"].map(geschuetzt) / ziel["plr_id"].map(plr_area)
).fillna(0.0).clip(upper=1.0)        

#variable 2 — binary majority flag (1 if >50% of PLR area protected)
ziel["milieu_majoritaet"] = (ziel["milieu_anteil"] > 0.5).astype(int)

#Inspect and check 
print("\n--- Continuous label (milieu_anteil) ---")
print(ziel["milieu_anteil"].describe())
print("PLR mit irgendeiner Abdeckung (>0):", int((ziel["milieu_anteil"] > 0).sum()))

print("\n--- Binary label (milieu_majoritaet, >50%) ---")
print(ziel["milieu_majoritaet"].value_counts())
print(f"Positivklasse: {ziel['milieu_majoritaet'].sum()} von {len(ziel)} PLR "
      f"({ziel['milieu_majoritaet'].mean()*100:.1f}%)")

#Save
import os
ziel.to_csv("../../data/Module 2/target_milieuschutz.csv", index=False)
print("\ngespeichert:", ziel.shape, "-> target_milieuschutz.csv")
ziel.head()

Milieuschutzgebiete geladen: (82, 10)
Spalten: ['gml_id', 'schluessel', 'bezirk', 'gebietsname', 'f_gvbl_dat', 'f_in_kraft', 'ae_gvbldat', 'ae_inkraft', 'fl_ha', 'geometry']
PLR geladen: 542

--- Continuous label (milieu_anteil) ---
count    542.000000
mean       0.189239
std        0.326469
min        0.000000
25%        0.000000
50%        0.000000
75%        0.255429
max        1.000000
Name: milieu_anteil, dtype: float64
PLR mit irgendeiner Abdeckung (>0): 218

--- Binary label (milieu_majoritaet, >50%) ---
milieu_majoritaet
0    433
1    109
Name: count, dtype: int64
Positivklasse: 109 von 542 PLR (20.1%)

gespeichert: (542, 5) -> target_milieuschutz.csv


,plr_id,plr_name,bez,milieu_anteil,milieu_majoritaet
0,01100101,Stülerstraße,01 - Mitte,0.000000e+00,0
1,01100102,Großer Tiergarten,01 - Mitte,0.000000e+00,0
2,01100103,Lützowstraße,01 - Mitte,7.967912e-01,1
3,01100104,Körnerstraße,01 - Mitte,8.400603e-01,1
4,01100205,Wilhelmstraße,01 - Mitte,2.602949e-08,0


In [3]:
# EDA 

lab = pd.read_csv("../../data/Module 2/target_milieuschutz.csv", dtype={"plr_id": str})
lab["plr_id"] = lab["plr_id"].astype(str).str.zfill(8)

print("rows:", len(lab), "| unique plr_id:", lab["plr_id"].nunique())
print("\nmissings per column:\n", lab.isna().sum().to_string())

#consistency: does majoritaet == (anteil > 0.5)? (checks the label derivation itself)
derived = (lab["milieu_anteil"] > 0.5).astype(int)
mismatch = (derived != lab["milieu_majoritaet"]).sum()
print(f"\nmajoritaet vs. (anteil > 0.5) mismatches: {mismatch}")

rows: 542 | unique plr_id: 542

missings per column:
 plr_id               0
plr_name             0
bez                  0
milieu_anteil        0
milieu_majoritaet    0

majoritaet vs. (anteil > 0.5) mismatches: 0


### Result Discussion 

Of 542 PLR, 218 carry some Milieuschutz coverage and 109 exceed the 50% majority threshold, giving a positive class of 20.1% — a substantial imbalance that the classification metrics and model weighting must account for. 

The continuous share is heavily zero-inflated (median 0, 324 PLR at exactly zero) and never reaches a full 1.0, reflecting that protection typically covers only part of a planning area. 

A consistency check confirms the binary label matches (milieu_anteil > 0.5) with zero mismatches, and no column has missing values, so the target table is complete across all 542 PLR. 

Crucially, this label encodes political designation decisions made independently by each district, not an objective measure of displacement. The model therefore learns similarity to designated areas, and confident false positives are analytically valuable rather than errors.